# Generate entity lists

This notebook has some quick methods for generate listings of chemicals, genes and diseases and saving them out.

In [33]:
import gzip
import requests
import re

def load_pubchem_synonyms(url):
    response = requests.get(url, stream=True)
    response.raise_for_status()

    chemicals = set()

    with gzip.GzipFile(fileobj=response.raw) as f:
        for line in f:
            parts = line.decode("utf-8").strip().split("\t")
            if len(parts) >= 2:
                synonym = parts[1]
                chemicals.add(synonym)

                if len(chemicals) >= 1000000:  # cap size
                    break

    return list(chemicals)


url = "https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/CID-Synonym-filtered.gz"
chemicals = load_pubchem_synonyms(url)

triple_number_regex = re.compile(r'\d\d\d')
chemicals = [ chemical for chemical in chemicals if not triple_number_regex.search(chemical) and len(chemical) < 50 and not ',' in chemical and not '[' in chemical ]
print(len(chemicals))
print(chemicals[:20])

262571
['a thiazole', '2-methyl-2-iodopropane', 'Atmurin', '3-FLUOROPROPYLENE OXIDE', 'Theraptique', '2-Phenyllepidine', 'Enteroseptol', 'SPARKLING MIST', '4-Hydroxybutanamide (RG)', 'Diphenyldiimide', 'N06AF01', 'N-methyl phthalimide', 'Acido flufenamico', 'ChromaSafe Hand Sanitizer', 'ACRYESTER EH', '9-Oxoxanthene', 'dichloro(ethyl)silane', 'Celliton Red B', 'DL-3-aminobutanoic acid', '5-Flucytosine']


In [34]:
# -----------------------------
# GENES / PROTEINS (NCBI Entrez)
# -----------------------------
def get_genes(limit=100):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    
    # Search for human genes
    search_url = f"{base}esearch.fcgi"
    params = {
        "db": "gene",
        "term": "Homo sapiens[Organism]",
        "retmax": limit,
        "retmode": "json"
    }
    
    search = requests.get(search_url, params=params).json()
    ids = search["esearchresult"]["idlist"]
    
    # Fetch summaries
    summary_url = f"{base}esummary.fcgi"
    summaries = requests.get(summary_url, params={
        "db": "gene",
        "id": ",".join(ids),
        "retmode": "json"
    }).json()
    
    genes = []
    for gid in ids:
        item = summaries["result"].get(gid, {})
        if item:
            genes.append(item.get("name"))
    
    return genes

genes = get_genes(500)
len(genes)

500

In [35]:
# pip install pronto

from pronto import Ontology

def get_doids_with_names(file_or_url="https://purl.obolibrary.org/obo/doid.obo"):
    ont = Ontology(file_or_url)
    results = []

    for term in ont.terms():
        if term.id.startswith("DOID:") and not term.obsolete:
            results.append(term.name)

    return results

diseases = get_doids_with_names()
len(diseases)

In [37]:
import json
output = {'chemical':chemicals, 'gene':genes, 'disease':diseases}
with open('terms.json','w') as f:
    json.dump(output,f,indent=2)